# Additional Baseline ML Models for Dementia-Risk Classification

Extend the initial baseline comparison by evaluating Decision Tree, K-Nearest Neighbors (KNN), and Extra Trees.

The earlier baseline notebook evaluated Logistic Regression, Random Forest, and SVM. This notebook adds different model families using the existing feature CSV, a reproducible stratified holdout split, and the same evaluation metrics.


## 1. Imports and load the existing feature CSV

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, make_scorer
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import ExtraTreesClassifier

RANDOM_STATE = 42
warnings.filterwarnings("ignore")

csv_candidates = [
    Path("analysis_output/baseline_audio_features.csv"),
    Path("../notebooks/analysis_output/baseline_audio_features.csv"),
    Path(r"D:\SKIT_Project\notebooks\analysis_output\baseline_audio_features.csv"),
]
csv_path = next((p.resolve() for p in csv_candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("baseline_audio_features.csv not found. Run baseline_ml_model.ipynb first or update csv_candidates.")

data = pd.read_csv(csv_path)
print("Loaded:", csv_path)
print("Dataset shape:", data.shape)
display(data.head())

Loaded: D:\SKIT_Project\notebooks\analysis_output\baseline_audio_features.csv
Dataset shape: (360, 31)


,mfcc_1_mean,mfcc_1_std,mfcc_2_mean,mfcc_2_std,mfcc_3_mean,mfcc_3_std,mfcc_4_mean,mfcc_4_std,mfcc_5_mean,mfcc_5_std,...,mfcc_11_std,mfcc_12_mean,mfcc_12_std,mfcc_13_mean,mfcc_13_std,duration_seconds,rms_mean,zcr_mean,path,label
0,-290.894226,90.324074,103.325874,40.952290,31.520834,27.468496,18.506176,24.015102,8.285114,17.901567,...,9.639206,3.368045,10.388342,-5.722221,11.232134,71.019688,0.042058,0.102158,D:\SKIT_Project\dataset\dementia\Abe Burrows\A...,dementia
1,-332.609802,80.299843,119.011658,42.997719,2.573827,32.044609,18.508610,22.891491,-3.082855,17.222090,...,10.582289,-9.440688,9.598793,5.068859,9.860040,47.000000,0.037931,0.104126,D:\SKIT_Project\dataset\dementia\Aileen Hernan...,dementia
2,-210.087753,119.215401,68.219551,46.933098,1.398461,29.873440,13.520696,24.507381,1.728840,19.050022,...,11.535592,-11.186091,11.773222,-5.553871,10.998050,38.000000,0.074112,0.142498,D:\SKIT_Project\dataset\dementia\Aileen Hernan...,dementia
3,-338.254486,84.265968,126.706917,42.934311,15.690623,33.033348,-8.187231,26.539669,10.898719,21.172417,...,10.341165,-9.548878,9.538286,-2.497692,8.797155,90.000000,0.031270,0.103313,D:\SKIT_Project\dataset\dementia\Aileen Hernan...,dementia
4,-361.912262,105.808426,96.568855,54.680119,2.274842,27.184465,36.884491,28.228430,4.056905,17.687374,...,13.294925,-4.440179,10.911772,-7.837333,10.456482,55.000000,0.018489,0.126244,D:\SKIT_Project\dataset\dementia\Alan Ramsey\a...,dementia


## 2. Check labels and feature columns

In [ ]:
if not {"path", "label"}.issubset(data.columns):
    raise ValueError("The feature CSV must contain 'path' and 'label' columns.")

feature_cols = [c for c in data.columns if c not in {"path", "label"}]
print("Feature count:", len(feature_cols))
print("Class counts:")
display(data["label"].value_counts(dropna=False).to_frame("count"))
print("Missing feature values:", int(data[feature_cols].isna().sum().sum()))
print("Duplicate audio paths:", int(data["path"].duplicated().sum()))

Feature count: 29
Class counts:


,count
label,
nodementia,229
dementia,131


Missing feature values: 0
Duplicate audio paths: 0


## 3. Create a reproducible stratified train/test split

In [3]:
label_map = {"nodementia": 0, "dementia": 1}
y_all = data["label"].astype(str).str.strip().str.lower().map(label_map)
valid = y_all.notna()
data = data.loc[valid].reset_index(drop=True)
y = y_all.loc[valid].astype(int).reset_index(drop=True)
X = data[feature_cols].apply(pd.to_numeric, errors="coerce")

if y.nunique() != 2:
    raise ValueError("Expected both nodementia and dementia labels. Check label values.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print("Feature matrix:", X.shape)
print("Training samples:", len(X_train), "| Test samples:", len(X_test))
print("Training class counts:", y_train.value_counts().to_dict())
print("Test class counts:", y_test.value_counts().to_dict())

Feature matrix: (360, 29)
Training samples: 270 | Test samples: 90
Training class counts: {0: 172, 1: 98}
Test class counts: {0: 57, 1: 33}


## 4. Define additional baseline models

In [10]:
models = {
    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(
            class_weight="balanced",
            random_state=RANDOM_STATE,
            min_samples_leaf=3
        ))
    ]),
    "Extra Trees": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            min_samples_leaf=2
        ))
    ]),
}

print("Models to evaluate:", list(models.keys()))

Models to evaluate: ['Decision Tree', 'Extra Trees']


## 5. Evaluate on the held-out test set

In [11]:
holdout_rows = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred
    holdout_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Dementia Precision": precision_score(y_test, pred, pos_label=1, zero_division=0),
        "Dementia Recall": recall_score(y_test, pred, pos_label=1, zero_division=0),
        "Dementia F1-score": f1_score(y_test, pred, pos_label=1, zero_division=0),
    })

holdout_results = pd.DataFrame(holdout_rows).sort_values(
    ["Dementia F1-score", "Dementia Recall"], ascending=False
).reset_index(drop=True)
display(holdout_results.round(3))

,Model,Accuracy,Dementia Precision,Dementia Recall,Dementia F1-score
0,Decision Tree,0.633,0.5,0.515,0.507
1,Extra Trees,0.667,0.6,0.273,0.375


## 6. Cross-validation on the training split

In [12]:
scoring = {
    "accuracy": "accuracy",
    "precision_dementia": make_scorer(precision_score, pos_label=1, zero_division=0),
    "recall_dementia": make_scorer(recall_score, pos_label=1, zero_division=0),
    "f1_dementia": make_scorer(f1_score, pos_label=1, zero_division=0),
}
n_splits = min(5, int(y_train.value_counts().min()))
if n_splits < 2:
    raise ValueError("Not enough training examples per class for cross-validation.")

cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    cv_rows.append({
        "Model": name,
        "CV Accuracy Mean": scores["test_accuracy"].mean(),
        "CV Accuracy Std": scores["test_accuracy"].std(),
        "CV Dementia Recall Mean": scores["test_recall_dementia"].mean(),
        "CV Dementia F1 Mean": scores["test_f1_dementia"].mean(),
    })

cv_results = pd.DataFrame(cv_rows).sort_values(
    ["CV Dementia F1 Mean", "CV Dementia Recall Mean"], ascending=False
).reset_index(drop=True)
display(cv_results.round(3))

,Model,CV Accuracy Mean,CV Accuracy Std,CV Dementia Recall Mean,CV Dementia F1 Mean
0,Decision Tree,0.581,0.034,0.418,0.405
1,Extra Trees,0.637,0.060,0.215,0.296


## 7. Classification reports and confusion matrices

In [ ]:
for name, pred in predictions.items():
    print("\\n" + "=" * 65)
    print(name)
    print(classification_report(
        y_test, pred, labels=[0, 1],
        target_names=["No Dementia", "Dementia"], zero_division=0
    ))
    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    plt.figure(figsize=(4.5, 3.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Predicted No Dementia", "Predicted Dementia"],
                yticklabels=["Actual No Dementia", "Actual Dementia"])
    plt.title(f"Confusion Matrix — {name}")
    plt.tight_layout()
    plt.show()

## 8. Save actual results

In [ ]:
output_dir = Path("analysis_output")
output_dir.mkdir(exist_ok=True)
holdout_path = output_dir / "additional_baseline_models_holdout.csv"
cv_path = output_dir / "additional_baseline_models_cv.csv"
holdout_results.to_csv(holdout_path, index=False)
cv_results.to_csv(cv_path, index=False)
print("Saved holdout results:", holdout_path.resolve())
print("Saved cross-validation results:", cv_path.resolve())